# 💰 SpendWise — Visual Analysis

This notebook connects directly to `spendwise.db` and visualises the data using Pandas, Matplotlib, and Seaborn.

Run `python main.py --seed` first to populate the database.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DB_PATH = os.path.join('..', 'spendwise.db')

def load(query, params=()):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(query, conn, params=params)
    conn.close()
    return df

print('Connected to:', os.path.abspath(DB_PATH))

## 1. Spending by Category

In [ ]:
df_cat = load("""
    SELECT c.name AS category, SUM(t.amount) AS total
    FROM transactions t
    JOIN categories c ON t.category_id = c.id
    WHERE c.type = 'expense'
    GROUP BY c.id
    ORDER BY total DESC
""")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_cat, x='total', y='category', ax=ax, palette='Blues_r')
ax.set_title('Total Spending by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Spent (€)')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
plt.tight_layout()
plt.show()

## 2. Monthly Income vs Expense Trend

In [ ]:
df_trend = load("""
    SELECT strftime('%Y-%m', t.date) AS month,
           SUM(CASE WHEN c.type = 'income'  THEN t.amount ELSE 0 END) AS income,
           SUM(CASE WHEN c.type = 'expense' THEN t.amount ELSE 0 END) AS expense
    FROM transactions t
    JOIN categories c ON t.category_id = c.id
    GROUP BY month
    ORDER BY month
""")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_trend['month'], df_trend['income'],  marker='o', label='Income',  linewidth=2)
ax.plot(df_trend['month'], df_trend['expense'], marker='s', label='Expense', linewidth=2)
ax.fill_between(df_trend['month'], df_trend['income'], df_trend['expense'], alpha=0.1)
ax.set_title('Monthly Income vs Expense', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Amount (€)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 3. Budget vs Actual (May 2026)

In [ ]:
MONTH = '2026-05'

df_bva = load("""
    SELECT c.name AS category,
           b.limit_amount AS budget,
           COALESCE(SUM(t.amount), 0) AS actual
    FROM budgets b
    JOIN categories c ON b.category_id = c.id
    LEFT JOIN transactions t
        ON t.category_id = b.category_id
        AND strftime('%Y-%m', t.date) = b.month
    WHERE b.month = ?
    GROUP BY b.id
    ORDER BY budget DESC
""", params=(MONTH,))

x = range(len(df_bva))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - width/2 for i in x], df_bva['budget'], width, label='Budget',  color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], df_bva['actual'], width, label='Actual',
               color=['tomato' if a > b else 'mediumseagreen'
                      for a, b in zip(df_bva['actual'], df_bva['budget'])])
ax.set_xticks(list(x))
ax.set_xticklabels(df_bva['category'], rotation=20, ha='right')
ax.set_title(f'Budget vs Actual — {MONTH}', fontsize=14, fontweight='bold')
ax.set_ylabel('Amount (€)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'€{v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()